# OpenDose-PopPK — Demo: Paracetamol
Population PK/PD Simulation with Covariate Modeling

**Author:** Angelo Gabriel C. Silva Gomes (IFB, 2026)

In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt

from opendose_poppk import (
    DrugDatabase, PKModel, PDModel,
    CovariateModel, PopulationSimulator, MAPEstimator,
    plot_monte_carlo, plot_population_with_covariates, plot_map_fit
)

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'font.size': 12, 'figure.dpi': 120})
print('Setup OK')

## 1. Load Drug Parameters

In [ ]:
db   = DrugDatabase('../datasets/drugs_parameters.csv')
drug = db.get_drug('Paracetamol')

print(f'Drug   : {drug.name}')
print(f'F      : {drug.F}')
print(f'ka     : {drug.ka} h⁻¹')
print(f'ke     : {drug.ke} h⁻¹')
print(f'Vd     : {drug.Vd} L')
print(f'EC50   : {drug.EC50} µg/mL')
print(f'n_hill : {drug.n_hill}')
print(f'Dose   : {drug.dose} mg')

## 2. PK/PD Models

In [ ]:
pk = PKModel(**drug.pk_kwargs)
pd = PDModel(drug.EC50, drug.n_hill)

# Analytical metrics
cmax, tmax = pk.cmax(D=drug.dose)
auc        = pk.auc(D=drug.dose)
ss         = pk.state_space()

print(f'Cmax   = {cmax:.2f} µg/mL  at  Tmax = {tmax:.2f} h')
print(f'AUC₀→∞ = {auc:.1f} µg·h/mL')
print(f'Eigenvalues: {ss["eigenvalues"]}  →  Stable: {ss["is_stable"]}')
print(f'EC90   = {pd.ec_x(0.9):.2f} µg/mL')

## 3. Monte Carlo Simulation (no covariates)

In [ ]:
fig = plot_monte_carlo(pk, pd, dose=drug.dose, t_max=12.0,
                       n_subjects=1000, drug_name=drug.name)
plt.show()

## 4. Covariate-Adjusted Population Simulation

In [ ]:
cov = CovariateModel(pk)
sim = PopulationSimulator(pk, pd, cov, dose=drug.dose)

result = sim.run(
    n_subjects=1000,
    t_max=12.0,
    seed=42,
    covariates={
        'weight': ('normal', 70.0, 15.0),   # kg
        'crcl':   ('normal', 90.0, 30.0),   # mL/min
        'age':    ('normal', 45.0, 15.0),   # years
    }
)

ppk = result['percentiles_pk']
print(f'Median Cmax : {ppk[50].max():.2f} µg/mL')
print(f'PI90  Cmax  : {ppk[5].max():.2f} – {ppk[95].max():.2f} µg/mL')

In [ ]:
fig = plot_population_with_covariates(result, drug.name, drug.dose)
plt.show()

## 5. MAP Estimation — Individual Patient

In [ ]:
# Observed samples from a real patient
t_obs = np.array([0.5, 1.0, 2.0, 4.0, 6.0, 8.0])   # h
c_obs = np.array([4.2, 6.8, 7.5, 5.9, 4.1, 2.8])   # µg/mL

# Patient covariates
patient = {'weight': 95.0, 'crcl': 45.0, 'age': 68.0}

# Fit
est = MAPEstimator(pk, covariate_model=cov, sigma_obs=0.8)
res = est.fit(t_obs, c_obs, patient, dose=drug.dose)

print(f'Converged: {res["converged"]}')
print(f'{"Param":6s}  {"Pop-adj":>12s}  {"MAP":>12s}  {"Eta":>8s}')
print('─' * 46)
for p in ('Vd', 'ke', 'ka', 'F'):
    print(f'{p:6s}  {res["pop_adjusted"][p]:12.4f}  '
          f'{res["params_map"][p]:12.4f}  {res["eta_map"][p]:+8.3f}')

In [ ]:
patient_str = f"weight={patient['weight']}kg, CrCl={patient['crcl']}mL/min, age={patient['age']}yr"
fig = plot_map_fit(pk, res, t_obs, c_obs,
                   dose=drug.dose, drug_name=drug.name,
                   patient_info=patient_str)
plt.show()

## 6. Adding a Custom Covariate

Example: serum albumin affects Vd for highly protein-bound drugs.

In [ ]:
# Add albumin as a new covariate
cov.add_covariate(
    name='albumin',
    reference=4.0,            # g/dL (normal value)
    betas={'Vd': 0.30}        # hypoalbuminemia increases Vd
)

# Simulate a hypoalbuminemic patient
p_low_alb = cov.individualize(
    covariates={'weight': 70.0, 'albumin': 2.5},  # low albumin
    sex='M'
)
p_normal  = cov.individualize(
    covariates={'weight': 70.0, 'albumin': 4.0},  # normal
    sex='M'
)

print(f'Normal albumin  → Vd = {p_normal["Vd"]:.1f} L')
print(f'Low albumin     → Vd = {p_low_alb["Vd"]:.1f} L')